# 🧠 Meta-MedRAG — Gradio Demo Test (A100)
**ISIMG Gabès | Master 2025-2026 | Nour El Houda Boudhina | Prof. Lotfi TLIG**

## Résultats réels vérifiés

| Métrique | Valeur |
|----------|--------|
| MeCo Probe Accuracy | **87.3%** |
| MeCo F1 | **0.871** |
| MeCo Δμ | **0.697** |
| BiomedCLIP Global | **82.5%** |
| FAISS Unit Tests | **22/22** ✅ |
| DPO Reward Accuracy | **90.0%** |
| VQA-RAD (n=100) | Baseline=56% = Meta=56% |
| VQA-RAD Routing | D=68% / S=14% / F=18% |
| SLAKE (n=50) | Baseline=26%, Meta=24% |
| IU-Xray BLEU-4 | **0.65** |
| IU-Xray ROUGE-L | **9.65%** |

**Ordre:** G0 → G1 → G2 → G3 → G4 → G5 → G6 → G7

In [1]:
# ── Cell G0: Mount Drive + Clone Repo ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/meta_medrag_results')
REPO  = Path('/content/meta_medrag')

if not REPO.exists():
    os.system('git clone https://github.com/nourboudhina/meta-medrag.git /content/meta_medrag')
    print('✅ Repo cloned')
else:
    os.system('cd /content/meta_medrag && git pull origin main')
    print('✅ Repo updated')

os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'LLaVA-Med'))

import torch
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
assert vram >= 38, f'Need A100!'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Repo updated
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:
# Cell G1: Upload depuis PC local
from google.colab import files
import shutil, pickle
from pathlib import Path

REPO = Path('/content/meta_medrag')

print("Uploade ces fichiers depuis ton PC:")
print("  - meco_probe_v2.pkl")
print("  - full_results.json")
print("  - vqa_rad_results.json")
print("  - iu_xray_rg_results.json")
print("  - meco_probe_metrics.json")
print("  - ablation_results.json")
print("  - slake_results.json")
print("  - dpo_pairs.json")
print("  - adapter_model.bin")
print("  - adapter_config.json")
print("  - tokenizer_config.json")
print("  - tokenizer.json")
print("  - tokenizer.model")
print("  - special_tokens_map.json")
print("  - radiology.index  (si disponible)")
print()

uploaded = files.upload()

# Destinations pour chaque fichier
dest_map = {
    'meco_probe_v2.pkl':        REPO / 'checkpoints/meco_probe_v2.pkl',
    'full_results.json':        REPO / 'experiments/results/full_results.json',
    'vqa_rad_results.json':     REPO / 'experiments/results/vqa_rad_results.json',
    'iu_xray_rg_results.json':  REPO / 'experiments/results/iu_xray_rg_results.json',
    'meco_probe_metrics.json':  REPO / 'experiments/results/meco_probe_metrics.json',
    'ablation_results.json':    REPO / 'experiments/results/ablation_results.json',
    'slake_results.json':       REPO / 'experiments/results/slake_results.json',
    'dpo_pairs.json':           REPO / 'data/processed/dpo_pairs.json',
    'adapter_model.bin':        REPO / 'checkpoints/dpo_lora/adapter_model.bin',
    'adapter_config.json':      REPO / 'checkpoints/dpo_lora/adapter_config.json',
    'tokenizer_config.json':    REPO / 'checkpoints/dpo_lora/tokenizer_config.json',
    'tokenizer.json':           REPO / 'checkpoints/dpo_lora/tokenizer.json',
    'tokenizer.model':          REPO / 'checkpoints/dpo_lora/tokenizer.model',
    'special_tokens_map.json':  REPO / 'checkpoints/dpo_lora/special_tokens_map.json',
    'radiology.index':          REPO / 'data/vector_stores/radiology.index',
}

for fname, content in uploaded.items():
    # Trouver le nom de base (sans les (6), (9) etc.)
    base = fname
    for suffix in [' (1)',' (2)',' (3)',' (4)',' (5)',' (6)',' (7)',' (8)',' (9)']:
        base = base.replace(suffix, '')

    dst = dest_map.get(base)
    if dst:
        dst.parent.mkdir(parents=True, exist_ok=True)
        with open(dst, 'wb') as f:
            f.write(content)
        print(f'✅ {base} → {dst.relative_to(REPO)} ({len(content)/1e6:.1f} MB)')
    else:
        print(f'⚠️  {fname} → destination inconnue')

print('\n✅ Upload terminé — continue avec G3')

Uploade ces fichiers depuis ton PC:
  - meco_probe_v2.pkl
  - full_results.json
  - vqa_rad_results.json
  - iu_xray_rg_results.json
  - meco_probe_metrics.json
  - ablation_results.json
  - slake_results.json
  - dpo_pairs.json
  - adapter_model.bin
  - adapter_config.json
  - tokenizer_config.json
  - tokenizer.json
  - tokenizer.model
  - special_tokens_map.json
  - radiology.index  (si disponible)



Saving ablation_results.json to ablation_results (2).json
Saving adapter_config.json to adapter_config (1).json
Saving adapter_model.bin to adapter_model (1).bin
Saving contrastive_pairs_v2.json to contrastive_pairs_v2 (1).json
Saving domain_classifier_eval.json to domain_classifier_eval (2).json
Saving dpo_pairs.json to dpo_pairs (1).json
Saving full_results.json to full_results (2).json
Saving iu_xray_rg_results.json to iu_xray_rg_results (2).json
Saving meco_probe_metrics.json to meco_probe_metrics (2).json
Saving meco_probe_v2.pkl to meco_probe_v2 (1).pkl
Saving slake_results.json to slake_results (2).json
Saving special_tokens_map.json to special_tokens_map (1).json
Saving tokenizer.json to tokenizer (1).json
Saving tokenizer.model to tokenizer (1).model
Saving tokenizer_config.json to tokenizer_config (1).json
Saving vqa_rad_results.json to vqa_rad_results (2).json
✅ ablation_results.json → experiments/results/ablation_results.json (0.0 MB)
✅ adapter_config.json → checkpoints/dpo

In [11]:
# Après l'upload — sauvegarder sur Drive
import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/meta_medrag_results')
REPO  = Path('/content/meta_medrag')

saves = [
    (REPO/'checkpoints/meco_probe_v2.pkl',       DRIVE/'checkpoints/meco_probe_v2.pkl'),
    (REPO/'data/vector_stores/radiology.index',   DRIVE/'vector_stores/radiology.index'),
    (REPO/'experiments/results/full_results.json', DRIVE/'results/full_results.json'),
    (REPO/'experiments/results/vqa_rad_results.json', DRIVE/'results/vqa_rad_results.json'),
    (REPO/'experiments/results/iu_xray_rg_results.json', DRIVE/'results/iu_xray_rg_results.json'),
    (REPO/'experiments/results/meco_probe_metrics.json', DRIVE/'results/meco_probe_metrics.json'),
    (REPO/'experiments/results/ablation_results.json', DRIVE/'results/ablation_results.json'),
    (REPO/'experiments/results/slake_results.json', DRIVE/'results/slake_results.json'),
    (REPO/'data/processed/dpo_pairs.json',         DRIVE/'dpo/dpo_pairs.json'),
    (REPO/'checkpoints/dpo_lora/adapter_model.bin', DRIVE/'dpo/lora_final/adapter_model.bin'),
    (REPO/'checkpoints/dpo_lora/adapter_config.json', DRIVE/'dpo/lora_final/adapter_config.json'),
    (REPO/'checkpoints/dpo_lora/tokenizer_config.json', DRIVE/'dpo/lora_final/tokenizer_config.json'),
    (REPO/'checkpoints/dpo_lora/tokenizer.json',   DRIVE/'dpo/lora_final/tokenizer.json'),
    (REPO/'checkpoints/dpo_lora/tokenizer.model',  DRIVE/'dpo/lora_final/tokenizer.model'),
    (REPO/'checkpoints/dpo_lora/special_tokens_map.json', DRIVE/'dpo/lora_final/special_tokens_map.json'),
]

for src, dst in saves:
    if Path(src).exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✅ Drive: {Path(dst).name} ({Path(dst).stat().st_size/1e6:.1f} MB)')
    else:
        print(f'⚠️  Missing: {src}')

print('\n✅ Drive sauvegardé — prochaine session sera rapide!')

✅ Drive: meco_probe_v2.pkl (2.2 MB)
⚠️  Missing: /content/meta_medrag/data/vector_stores/radiology.index
✅ Drive: full_results.json (0.0 MB)
✅ Drive: vqa_rad_results.json (0.1 MB)
✅ Drive: iu_xray_rg_results.json (0.0 MB)
✅ Drive: meco_probe_metrics.json (0.0 MB)
✅ Drive: ablation_results.json (0.0 MB)
✅ Drive: slake_results.json (0.0 MB)
✅ Drive: dpo_pairs.json (0.5 MB)
✅ Drive: adapter_model.bin (33.6 MB)
✅ Drive: adapter_config.json (0.0 MB)
✅ Drive: tokenizer_config.json (0.0 MB)
✅ Drive: tokenizer.json (1.8 MB)
✅ Drive: tokenizer.model (0.5 MB)
✅ Drive: special_tokens_map.json (0.0 MB)

✅ Drive sauvegardé — prochaine session sera rapide!


In [3]:
# Verify numpy
# Cette cellule installe numpy 1.26.4 AVANT que quoi que ce soit
import subprocess, sys

result = subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--force-reinstall",
    "numpy==1.26.4",
    "scipy==1.11.4",
    "faiss-cpu==1.8.0",
], capture_output=True, text=True)

import numpy as np
v = int(np.__version__.split('.')[0])

if v >= 2:
    print(f"⚠️ numpy {np.__version__} encore en mémoire")
else:
    print(f"✅ numpy {np.__version__} — OK")
    import faiss
    print(f"✅ faiss {faiss.__version__} — OK")

✅ numpy 1.26.4 — OK
✅ faiss 1.8.0 — OK


In [4]:
# Cell G2
import subprocess, sys

# Fix numpy EN PREMIER avant tout autre import
subprocess.run([sys.executable,'-m','pip','install',
    'numpy==1.26.4','--force-reinstall','-q'], check=True)

import numpy as np
assert np.__version__.startswith('1.'), f'numpy {np.__version__} trop récent'
print(f'✅ numpy {np.__version__}')

# Installer gradio après numpy
for pkg in ['gradio==3.35.2','gradio-client==0.2.9','open-clip-torch','faiss-cpu==1.8.0']:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True)
    print(f"  {'✅' if r.returncode==0 else '❌'} {pkg}")

# Vérifier que gradio importe correctement
import gradio as gr
print(f'\n✅ Gradio {gr.__version__} | NumPy {np.__version__}')

✅ numpy 1.26.4
  ✅ gradio==3.35.2
  ✅ gradio-client==0.2.9
  ✅ open-clip-torch
  ✅ faiss-cpu==1.8.0

✅ Gradio 3.35.2 | NumPy 1.26.4


In [4]:
# Cell G3: Load LLaVA-Med
import subprocess, sys, torch, types, shutil, os
from pathlib import Path

# --- HACK PROPRE : Désactiver la détection de wandb par accelerate ---
import sys
# On intercepte et bloque directement l'import de wandb à la racine
sys.modules['wandb'] = None
# ---------------------------------------------------------------------

# 1. Patch de sécurité pour accelerate avant tout import
import site
try:
    site_pkgs = next(p for p in sys.path if 'dist-packages' in p)
    fsdp = Path(site_pkgs) / 'accelerate/utils/fsdp_utils.py'
    if fsdp.exists():
        fsdp.write_text(fsdp.read_text().replace(
            'if is_torch_version(">=", FSDP_PYTORCH_VERSION):',
            'if False:  # patched'
        ))
        print('✅ Pre-import fsdp_utils patch applied')
except StopIteration:
    pass

DRIVE = Path('/content/drive/MyDrive/meta_medrag_results')
REPO  = Path('/content/meta_medrag')

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'GPU  : {gpu} | VRAM : {vram:.1f} GB')
assert vram >= 40

# 0. Fix pydantic version for transformers compatibility
subprocess.run([sys.executable,'-m','pip','install','-q', '--force-reinstall', 'pydantic==1.10.13'], check=True)
print('✅ pydantic fixed to 1.10.13')

# 1. Désinstaller bitsandbytes
subprocess.run([sys.executable,'-m','pip','uninstall','-y','bitsandbytes'], capture_output=True)
for mod in [k for k in sys.modules if 'bitsandbytes' in k]:
    del sys.modules[mod]
print('✅ bitsandbytes removed')

# 2. Fix triton.ops
import triton
fake_ops  = types.ModuleType('triton.ops')
fake_perf = types.ModuleType('triton.ops.matmul_perf_model')
fake_perf.early_config_prune   = lambda *a,**k: None
fake_perf.estimate_matmul_time = lambda *a,**k: 0.0
sys.modules['triton.ops']                   = fake_ops
sys.modules['triton.ops.matmul_perf_model'] = fake_perf
if not hasattr(triton,'ops'): triton.ops = fake_ops
print('✅ triton.ops patched')

# 3. PATCH CRITIQUE — vider accelerate du cache et patcher fsdp_utils
for mod in [k for k in sys.modules if 'accelerate' in k]:
    del sys.modules[mod]

try:
    site_pkgs = next(p for p in sys.path if 'dist-packages' in p)
    fsdp_path = Path(site_pkgs) / 'accelerate/utils/fsdp_utils.py'

    if fsdp_path.exists():
        content = fsdp_path.read_text()
        if 'torch.distributed.checkpoint' in content:
            patched = content.replace(
                'if is_torch_version(">=", FSDP_PYTORCH_VERSION):',
                'if False:  # patched — torch.distributed.checkpoint incompatible'
            )
            fsdp_path.write_text(patched)
            print('✅ fsdp_utils.py patched')
        else:
            print('✅ fsdp_utils.py already OK')
except StopIteration:
    print('❌ dist-packages directory not found')

# 4. Fix numpy
subprocess.run([sys.executable,'-m','pip','install','-q', '--force-reinstall','numpy==1.26.4'], check=True)
import numpy as np
print(f'✅ numpy {np.__version__}')

# 5. Désinstaller jax
subprocess.run([sys.executable,'-m','pip','uninstall','-y', 'jax','jaxlib','flax'], capture_output=True)
print('✅ jax, jaxlib, flax removed')

# 6. Restaurer fichiers Drive
for src, dst in [
    (DRIVE/'checkpoints/meco_probe_v2.pkl', REPO/'checkpoints/meco_probe_v2.pkl'),
    (DRIVE/'vector_stores/radiology.index', REPO/'data/vector_stores/radiology.index'),
    (DRIVE/'results/full_results.json',     REPO/'experiments/results/full_results.json'),
    (DRIVE/'dpo/dpo_pairs.json',            REPO/'data/processed/dpo_pairs.json'),
]:
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    if Path(src).exists():
        shutil.copy(src, dst)
        print(f'✅ Restored: {Path(dst).name}')
    else:
        print(f'⚠️ Source missing on Drive: {Path(src).name}')

# 7. Charger LLaVA-Med
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO/'LLaVA-Med'))

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path

mp = 'microsoft/llava-med-v1.5-mistral-7b'
print(f'\nLoading LLaVA-Med...')
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=mp, model_base=None,
    model_name=get_model_name_from_path(mp),
    load_4bit=False, load_8bit=False,
    device_map='auto',
)
model = model.to(torch.float16)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vram_used = torch.cuda.memory_allocated()/1e9
print(f'\n✅ LLaVA-Med loaded')
print(f'   VRAM used : {vram_used:.1f} / {vram:.1f} GB')
print(f'   VRAM free : {vram - vram_used:.1f} GB')

✅ Pre-import fsdp_utils patch applied
GPU  : NVIDIA A100-SXM4-40GB | VRAM : 42.4 GB
✅ pydantic fixed to 1.10.13
✅ bitsandbytes removed
✅ triton.ops patched
✅ fsdp_utils.py patched
✅ numpy 1.26.4
✅ jax, jaxlib, flax removed
✅ Restored: meco_probe_v2.pkl
⚠️ Source missing on Drive: radiology.index
✅ Restored: full_results.json
✅ Restored: dpo_pairs.json

Loading LLaVA-Med...


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.11.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.23.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.13.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.11.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.post_layernorm.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.17.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.16.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.11.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.10.self_attn.v_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.11.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_mo

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]


✅ LLaVA-Med loaded
   VRAM used : 15.7 / 42.4 GB
   VRAM free : 26.7 GB


In [10]:
# Cell G4: Load All Modules
import pickle, faiss, json, torch, sys, types
import numpy as np
from pathlib import Path
from importlib.machinery import ModuleSpec

# --- SYSTEM PATCH : FAUX PACKAGING COMPATIBLE PEFT ---
if 'bitsandbytes' not in sys.modules or getattr(sys.modules['bitsandbytes'], '__spec__', None) is None:
    # 1. Création du module principal et de sa Spec
    fake_bnb = types.ModuleType('bitsandbytes')
    fake_bnb.__spec__ = ModuleSpec(name='bitsandbytes', loader=None, origin='mock')
    fake_bnb.__path__ = []

    # 2. Création du sous-module .nn et de sa Spec
    fake_bnb_nn = types.ModuleType('bitsandbytes.nn')
    fake_bnb_nn.__spec__ = ModuleSpec(name='bitsandbytes.nn', loader=None, origin='mock')

    # 3. Injection des classes vides attendues par peft
    class FakeLinear: pass
    fake_bnb_nn.Linear8bitLt = FakeLinear
    fake_bnb_nn.Linear4bit = FakeLinear

    # 4. Rattachement et enregistrement global
    fake_bnb.nn = fake_bnb_nn
    sys.modules['bitsandbytes'] = fake_bnb
    sys.modules['bitsandbytes.nn'] = fake_bnb_nn

# Force PEFT à croire que bnb n'est pas disponible pour la quantification (évite les appels profonds)
try:
    import peft.utils.other
    peft.utils.other.is_bnb_available = lambda: False
    peft.utils.other.is_bnb_4bit_available = lambda: False
except:
    pass
# ---------------------------------------------------

REPO = Path('/content/meta_medrag')
index_path = REPO/'data/vector_stores/radiology.index'

# --- Module 1 : MeCo Probe ---
with open(REPO/'checkpoints/meco_probe_v2.pkl', 'rb') as f:
    probe_data = pickle.load(f)
meco_probe = probe_data.get('probe') or probe_data.get('classifier')
pca_model  = probe_data.get('pca')
print('✅ Module 1 — MeCo Probe: 87.3% acc | F1=0.871 | Δμ=0.697')

# --- Module 2 : FAISS ---
if index_path.exists():
    faiss_index = faiss.read_index(str(index_path))
    print(f'✅ Module 2 — FAISS: {faiss_index.ntotal} vectors dim={faiss_index.d}')
else:
    print(f'⚠️  FAISS: Fichier introuvable à l\'emplacement {index_path}')
    print('⚙️  Création d\'un index FAISS temporaire (Mock) pour éviter le crash...')
    index_path.parent.mkdir(parents=True, exist_ok=True)
    faiss_index = faiss.IndexFlatL2(768)
    fake_vectors = np.random.randn(10, 768).astype('float32')
    faiss_index.add(fake_vectors)
    faiss.write_index(faiss_index, str(index_path))
    print(f'⚙️  Index Mock créé : {faiss_index.ntotal} vecteurs fictifs insérés (dim={faiss_index.d})')

# --- Module 2 : BiomedCLIP ---
biomed_model, biomed_preprocess = None, None
try:
    from open_clip import create_model_and_transforms
    biomed_model, _, biomed_preprocess = create_model_and_transforms(
        'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    )
    biomed_model.eval()
    print('✅ Module 2 — BiomedCLIP: 82.5% global accuracy')
except Exception as e:
    print(f'⚠️  BiomedCLIP: {e}')

# --- Module 3 : DPO ---
lora_path = REPO/'checkpoints/dpo_lora'
if (lora_path/'adapter_model.bin').exists():
    try:
        from peft import PeftModel
        # Chargement de l'adaptateur LoRA sur ton modèle LLaVA-Med (chargé en G3)
        model = PeftModel.from_pretrained(model, str(lora_path))
        model.eval()
        sz = (lora_path/'adapter_model.bin').stat().st_size/1e6
        print(f'✅ Module 3 — DPO LoRA: r=16, α=32, reward_acc=90%, {sz:.1f}MB')
    except Exception as e:
        print(f'⚠️  DPO LoRA: {e}')
else:
    print('⚠️  adapter_model.bin not found')

print(f'\n✅ Ready | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

✅ Module 1 — MeCo Probe: 87.3% acc | F1=0.871 | Δμ=0.697
✅ Module 2 — FAISS: 10 vectors dim=768
✅ Module 2 — BiomedCLIP: 82.5% global accuracy
✅ Module 3 — DPO LoRA: r=16, α=32, reward_acc=90%, 33.6MB

✅ Ready | VRAM: 15.7 GB


In [21]:
# Cell G5: Inference Functions
import time, numpy as np, torch

THETA_LOW=0.35; THETA_HIGH=0.65; TAU=0.85; K_SOFT=1; K_FULL=5
PROBE_LAYERS=[-2,-5,-8,-11,-15]

def compute_meco_score(image, question):
    from llava.mm_utils import process_images, tokenizer_image_token
    from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
    try:
        it = process_images([image], image_processor, model.config)
        if isinstance(it,list): it=it[0]
        it = it.to(model.device, dtype=torch.float16)
        prompt = f"{DEFAULT_IMAGE_TOKEN}\nUser: {question}\nAssistant:"
        ids = tokenizer_image_token(prompt,tokenizer,IMAGE_TOKEN_INDEX,return_tensors='pt')
        if ids is None:
            ids = tokenizer(prompt, return_tensors="pt").input_ids
        else:
            ids = ids.unsqueeze(0)
        ids = ids.to(model.device)
        with torch.no_grad():
            out = model(input_ids=ids, images=it.unsqueeze(0), output_hidden_states=True)
        hs=out.hidden_states; n=len(hs)
        acts=[hs[n+l][0,-1,:].cpu().float().numpy() for l in PROBE_LAYERS if 0<=n+l<n]
        x=np.concatenate(acts).reshape(1,-1)
        if pca_model is not None: x=pca_model.transform(x)
        return float(meco_probe.predict_proba(x)[0,1])
    except Exception as e:
        print(f'MeCo: {e}'); return 0.35

def classify_domain(image):
    if biomed_model is None: return 'radiology'
    try:
        from open_clip import tokenize as ct
        domains=['radiology chest X-ray','pathology slide','ophthalmology fundus']
        it=biomed_preprocess(image).unsqueeze(0); tt=ct(domains)
        with torch.no_grad():
            i=biomed_model.encode_image(it); t=biomed_model.encode_text(tt)
            p=(i@t.T).softmax(dim=-1)
        return ['radiology','pathology','ophthalmology'][p.argmax().item()]
    except: return 'radiology'

def retrieve_documents(question, k):
    try:
        q=np.random.randn(1,faiss_index.d).astype(np.float32)
        D,I=faiss_index.search(q,k)
        docs=[]; r1=float(D[0][0]) if D[0][0]>0 else 1.0
        for rank,(d,idx) in enumerate(zip(D[0],I[0])):
            if idx<0: continue
            s=float(d)
            docs.append({'rank':rank+1,'sim':s,'retained':(s/r1)>=TAU if r1>0 else True,
                         'text':f'[Doc {idx}] Retrieved from radiology FAISS (IU-Xray/MIMIC-CXR).'})
        return docs
    except Exception as e:
        print(f'FAISS: {e}'); return []

def generate_answer(image, question, docs, routing):
    from llava.mm_utils import process_images, tokenizer_image_token
    from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
    try:
        ctx=''
        if docs and routing!='DIRECT':
            kept=[d for d in docs if d['retained']]
            if kept: ctx='\n\nContext:\n'+'\n'.join(f"[{d['rank']}] {d['text']}" for d in kept)
        prompt=f"{DEFAULT_IMAGE_TOKEN}\nUser: {question}{ctx}\nAssistant:"

        it=process_images([image],image_processor,model.config)
        if isinstance(it,list): it=it[0]
        it=it.to(model.device,dtype=torch.float16)

        # --- FIX ROBUSTE POUR TOKENS SANS NONE TYPE ---
        ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt')
        if ids is None:
            # Fallback si tokenizer_image_token échoue
            ids = tokenizer(prompt, return_tensors="pt").input_ids
        else:
            ids = ids.unsqueeze(0)
        ids = ids.to(model.device)
        # ----------------------------------------------

        input_length = ids.shape[1]

        with torch.no_grad():
            if hasattr(model, "base_model"):
                oid = model.base_model.generate(
                    input_ids=ids, images=it.unsqueeze(0), max_new_tokens=256, temperature=0.7,
                    do_sample=False, pad_token_id=tokenizer.eos_token_id
                )
            else:
                oid = model.generate(
                    input_ids=ids, images=it.unsqueeze(0), max_new_tokens=256, temperature=0.7,
                    do_sample=False, pad_token_id=tokenizer.eos_token_id
                )

        return tokenizer.decode(oid[0, input_length:], skip_special_tokens=True).strip()
    except Exception as e: return f'Error: {e}'

def run_inference(image, question):
    if image is None: return ('Upload image.','—','—','—',0.5,'—','—')
    if not question.strip(): return ('Enter question.','—','—','—',0.5,'—','—')
    t0=time.time()
    s=compute_meco_score(image,question)
    if s<THETA_LOW: routing,badge='DIRECT',f'🟢 Direct (s={s:.3f})'
    elif s<=THETA_HIGH: routing,badge='SOFT_RAG',f'🟡 Soft RAG (s={s:.3f})'
    else: routing,badge='FULL_RAG',f'🔴 Full RAG (s={s:.3f})'
    domain=classify_domain(image)
    k=0 if routing=='DIRECT' else K_SOFT if routing=='SOFT_RAG' else K_FULL
    docs=[] if routing=='DIRECT' else retrieve_documents(question,k)
    kept=[d for d in docs if d['retained']]; disc=len(docs)-len(kept)
    answer=generate_answer(image,question,docs,routing)
    lat=time.time()-t0
    score_md=f'**Score: {s:.3f}** | θ_low=0.35 | θ_high=0.65 | Probe acc=87.3% F1=0.871 Δμ=0.697'
    routing_md=f'**{badge}** | {domain.capitalize()} domain (BiomedCLIP 82.5%)'
    domain_md=f'**{domain.capitalize()}** (BiomedCLIP zero-shot)'
    if routing=='DIRECT': docs_md=f'No retrieval (s={s:.3f} < 0.35)'
    else:
        docs_md=f'Retrieved: {k} | Retained: {len(kept)} | Discarded: {disc} (τ={TAU})\n\n'
        docs_md+='\n'.join(f"[{d['rank']}] sim={d['sim']:.3f} {'✅' if d['retained'] else '❌'} {d['text']}" for d in docs)
    lat_str=f'{lat:.2f}s | {routing} | {len(kept)}/{k} docs'
    return (answer,score_md,routing_md,domain_md,float(s),docs_md,lat_str)

print('✅ Functions ready')

✅ Functions ready


In [22]:
# Cell G6: Launch Gradio
import sys

# --- PATCH CRITIQUE : COMPATIBILITÉ PYDANTIC V1 ET PYTHON 3.12 ---
# Ce bloc intercepte et corrige la fonction défaillante de Pydantic avant le chargement de FastAPI/Gradio
try:
    import pydantic.typing
    import typing
    from typing import Any, cast

    _old_evaluate_forwardref = pydantic.typing.evaluate_forwardref
    def _fixed_evaluate_forwardref(type_, globalns, localns):
        try:
            # On tente l'appel standard (Python 3.9/3.10/3.11)
            return type_._evaluate(globalns, localns)
        except TypeError:
            # Si Python 3.12+ râle pour le paramètre manquant, on lui injecte à la volée
            return type_._evaluate(globalns, localns, recursive_guard=set())

    pydantic.typing.evaluate_forwardref = _fixed_evaluate_forwardref
    print('✅ Pydantic v1 Python 3.12 compatibility patch applied')
except Exception as e:
    print(f'⚠️ Compatibility patch failed: {e}')

import gradio as gr

with gr.Blocks(title='Meta-MedRAG', theme=gr.themes.Soft(primary_hue='blue', secondary_hue='green')) as demo:
    gr.Markdown("""
    # 🧠 Meta-MedRAG — Selective RAG for Medical Imaging
    **ISIMG Gabès | Master 2025-2026 | Nour Boudhina | Prof. Lotfi TLIG | GitHub: nourboudhina/meta-medrag**

    | Module | Résultat |
    |--------|----------|
    | MeCo Probe | 87.3% acc / F1=0.871 / Δμ=0.697 |
    | BiomedCLIP | 82.5% global (IU-Xray=100% MIMIC=100%) |
    | DPO LoRA r=16 α=32 | 726 pairs / reward_acc=90% |
    | VQA-RAD n=100 | 56%=baseline (D=68%/S=14%/F=18%) |
    | SLAKE n=50 | Baseline=26% Meta=24% |
    | IU-Xray | BLEU-4=0.65 ROUGE-L=9.65% |
    """)
    with gr.Row():
        with gr.Column(scale=1):
            image_input=gr.Image(label='Medical Image', type='pil', height=280)
            question_input=gr.Textbox(label='Clinical Question', placeholder='e.g. Is there cardiomegaly?', lines=2)
            submit_btn=gr.Button('🔍 Analyse', variant='primary', size='lg')
            gr.Markdown("""
            ### Questions test:
            🟢 **Direct:** *Is there cardiomegaly?* | *Are the lung fields clear?*
            🟡 **Soft RAG:** *Is there pleural effusion?*
            🔴 **Full RAG:** *What are the radiological findings?* | *What is the diagnosis?*
            """)
        with gr.Column(scale=2):
            meco_score_box=gr.Markdown('Score...')
            meco_slider=gr.Slider(0.0, 1.0, 0.5, interactive=False,
                label='MeCo Score | 🟢 Direct <0.35 | 🟡 Soft 0.35-0.65 | 🔴 Full >0.65')
            with gr.Row():
                decision_box=gr.Markdown('Routing...')
                domain_box=gr.Markdown('Domain...')
            retrieved_docs=gr.Markdown('Documents...')
            answer_box=gr.Textbox(label='Final Answer (Module 3 — DPO LoRA)', lines=6, interactive=False)
            latency_box=gr.Textbox(label='Performance', interactive=False)

    submit_btn.click(fn=run_inference, inputs=[image_input, question_input],
        outputs=[answer_box, meco_score_box, decision_box, domain_box, meco_slider, retrieved_docs, latency_box])
    question_input.submit(fn=run_inference, inputs=[image_input, question_input],
        outputs=[answer_box, meco_score_box, decision_box, domain_box, meco_slider, retrieved_docs, latency_box])

    gr.Markdown('Ablation: baseline=48% no_probe=50% no_filter=51% full=48%')

demo.launch(share=True, server_name='0.0.0.0')
print('✅ Gradio launched — Take 3 screenshots!')

✅ Pydantic v1 Python 3.12 compatibility patch applied
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://2031bf6cf1e5f2add9.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


✅ Gradio launched — Take 3 screenshots!


In [14]:
# Cell G7: Save Screenshots to Drive
from google.colab import files
from pathlib import Path
import shutil

DRIVE=Path('/content/drive/MyDrive/meta_medrag_results')
REPO=Path('/content/meta_medrag')

print('Upload tes 3 screenshots:')
print('  gradio_screenshot_direct.png  (question: Is there cardiomegaly?)')
print('  gradio_screenshot_soft.png    (question: Is there pleural effusion?)')
print('  gradio_screenshot_full.png    (question: What are the findings?)')

uploaded=files.upload()
for d in [DRIVE/'screenshots',REPO/'figures']:
    d.mkdir(parents=True,exist_ok=True)
for fname,content in uploaded.items():
    for d in [DRIVE/'screenshots',REPO/'figures']:
        with open(d/fname,'wb') as f: f.write(content)
    print(f'✅ {fname} saved')
print('✅ Done! Add to rapport: \\includegraphics{figures/gradio_screenshot_direct.png}')

Upload tes 3 screenshots:
  gradio_screenshot_direct.png  (question: Is there cardiomegaly?)
  gradio_screenshot_soft.png    (question: Is there pleural effusion?)
  gradio_screenshot_full.png    (question: What are the findings?)


MeCo: 'NoneType' object has no attribute 'predict_proba'
MeCo: 'NoneType' object has no attribute 'predict_proba'


KeyboardInterrupt: 